In [1]:
import math
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
import time
import pandas as pd
from datetime import datetime, timedelta
import openpyxl
from difflib import SequenceMatcher
import string
import re

# Xử lí dữ liệu ngày sinh
def send_date_exact(element_name, value):
    element = driver.find_element(By.NAME, element_name)
    element.clear()
    time.sleep(0.1)
    if value:
        for char in value:
            element.send_keys(char)
            time.sleep(0.03)
        time.sleep(0.2)
        actual_value = element.get_attribute("value")
        if actual_value != value:
            script = f"""
            var element = document.querySelector('input[name="{element_name}"]');
            if (element) {{
                element.value = '{value}';
                element.setAttribute('value', '{value}');
                ['input', 'change', 'blur', 'keyup'].forEach(eventType => 
                    element.dispatchEvent(new Event(eventType, {{ bubbles: true }})));
            }}
            """
            driver.execute_script(script)
            time.sleep(0.1)

# KIỂM THỬ MÀN HÌNH ĐĂNG KÝ
def run_signin_tests():
    global driver
    wb = openpyxl.load_workbook("testdata_automation.xlsx")
    sheet = wb["testdata_automation_signin"]
    headers = [cell.value.strip() for cell in sheet[1] if cell.value]

    date_col = headers.index("Ngày sinh") + 1
    data = []
    for row_num, row in enumerate(sheet.iter_rows(min_row=2), start=2):
        row_data = [cell.value for cell in row]
        dict_row = dict(zip(headers, row_data))
        date_cell = sheet.cell(row_num, date_col)
        if date_cell.value is None:
            dict_row["Ngày sinh"] = ""
        elif isinstance(date_cell.value, str):
            dict_row["Ngày sinh"] = date_cell.value.strip()
        else:
            number_format = date_cell.number_format
            sep = '-' if '-' in number_format else '/'
            try:
                if isinstance(date_cell.value, datetime):
                    date_value = date_cell.value
                else:
                    base = datetime(1899, 12, 30)
                    date_value = base + timedelta(days=float(date_cell.value))
                dict_row["Ngày sinh"] = date_value.strftime(f'%d{sep}%m{sep}%Y')
            except:
                dict_row["Ngày sinh"] = str(date_cell.value)
        data.append(dict_row)
    data_df = pd.DataFrame(data)

    field_map = {
        "ho": "Họ",
        "ten": "Tên",
        "gioitinh": "Giới tính",
        "ngaysinh": "Ngày sinh",
        "diachi": "Địa chỉ",
        "email": "Email",
        "dienthoai": "SĐT",
        "username": "Username",
        "password": "Mật khẩu",
        "repassword": "Nhập lại mật khẩu"
    }

    testcase_wb = openpyxl.load_workbook("testcase.xlsx")
    sign_in_sheet = testcase_wb["Sign_in"]

    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.maximize_window()

    for i, row in data_df.iterrows():
        tc_id = row['Test Case ID']
        if pd.isna(tc_id):
            continue

        try:
            driver.get("http://127.0.0.1:8000/")
            time.sleep(1)
            driver.find_element(By.XPATH, "//*[@class='btn btn-success']").click()
            time.sleep(1)

            # Nhập form
            fields = [("ho", "Họ"), ("ten", "Tên")]
            for name, key in fields:
                value = row.get(key)
                if value and str(value).strip():
                    driver.find_element(By.NAME, name).send_keys(str(value).strip())

            gioitinh = row.get("Giới tính")
            if gioitinh and str(gioitinh).lower() in ["nam", "nữ", "khác"]:
                select = Select(driver.find_element(By.NAME, "gioitinh"))
                select.select_by_visible_text(gioitinh.capitalize())

            date_value = row.get("Ngày sinh")
            if date_value and str(date_value).strip():
                send_date_exact("ngaysinh", str(date_value).strip())

            fields = [("diachi", "Địa chỉ"), ("email", "Email"),
                      ("dienthoai", "SĐT"), ("username", "Username"), ("password", "Mật khẩu"),
                      ("repassword", "Nhập lại mật khẩu")]
            for name, key in fields:
                value = row.get(key)
                if value and str(value).strip():
                    driver.find_element(By.NAME, name).send_keys(str(value).strip())

            # Nhấn nút Đăng ký
            driver.find_element(By.XPATH, "//*[@class='form-control form-control-sm btn btn-primary']").click()
            time.sleep(2)

            # ==== Đọc kết quả từ màn hình ====
            actual_result = ""

            page = driver.page_source
            backend_error_keywords = ["SQLSTATE", "QueryException", "Whoops", "ErrorException", "Fatal error"]
            if any(keyword in page for keyword in backend_error_keywords):
                match = re.search(r"(SQLSTATE[^\<]+)", page, re.IGNORECASE | re.DOTALL)
                if match:
                    actual_result = match.group(1).strip()
                else:
                    match = re.search(r'class="exception-message">([^<]+)', page, re.DOTALL)
                    if match:
                        actual_result = match.group(1).strip()
                    else:
                        actual_result = re.sub(r'<[^>]+>', '', page[:500]).strip()

            possible_selectors = [
                ".alert-danger", ".invalid-feedback", ".text-danger", ".error", "[id*=error]",
                ".alert-success", ".text-success", ".success", "[id*=success]",
                ".invalid-feedback", "span.text-danger", "div.text-danger", "small.text-danger"]
            for selector in possible_selectors:
                try:
                    elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    for el in elements:
                        if el.is_displayed() and el.text.strip():
                            actual_result += el.text.strip() + " | "
                except:
                    continue

            try:
                error_lists = driver.find_elements(By.TAG_NAME, "ul")
                for ul in error_lists:
                    if ul.is_displayed():
                        lis = ul.find_elements(By.TAG_NAME, "li")
                        for li in lis:
                            if li.text.strip():
                                actual_result += li.text.strip() + " | "
            except:
                pass

            # Trim and clean
            actual_result = re.sub(r'\s+\|', '|', actual_result)
            actual_result = actual_result.strip(" | ")

            # ==== Cập nhật kết quả vào sheet ====
            found = False
            for r in range(1, sign_in_sheet.max_row + 1):
                cell_value = sign_in_sheet.cell(r, 1).value
                if str(cell_value).strip() == str(tc_id).strip():
                    sign_in_sheet.cell(r, 6).value = actual_result
                    expected = sign_in_sheet.cell(r, 5).value
                    if compare_actual_expected(actual_result, expected):
                        sign_in_sheet.cell(r, 7).value = "Pass"
                    else:
                        sign_in_sheet.cell(r, 7).value = "Fail"
                    found = True
                    break

            if not found:
                print(f"TC ID {tc_id} not found in testcase.xlsx sheet Sign_in.")

        except Exception as e:
            actual_result = f"Exception during test: {str(e)}"
            for r in range(1, sign_in_sheet.max_row + 1):
                if str(sign_in_sheet.cell(r, 1).value).strip() == str(tc_id).strip():
                    sign_in_sheet.cell(r, 6).value = actual_result
                    sign_in_sheet.cell(r, 7).value = "Fail"
                    break

    testcase_wb.save("testcase.xlsx")
    driver.quit()
    print("Kiểm thử đăng ký hoàn tất. Đã cập nhật Kết quả thực sự và Trạng thái vào file testcase.xlsx.")

# KIỂM THỬ ĐĂNG NHẬP
def run_login_tests():
    # Đọc file Excel và xử lý dữ liệu
    wb = openpyxl.load_workbook("testdata_automation.xlsx")
    sheet = wb["testdata_automation_login"]
    headers = [cell.value.strip() for cell in sheet[1] if cell.value]
    data = []
    for row in sheet.iter_rows(min_row=2):
        row_data = {headers[i]: cell.value for i, cell in enumerate(row) if i < len(headers)}
        data.append(row_data)
    data_df = pd.DataFrame(data)

    # Khởi tạo driver và thực thi test
    testcase_wb = openpyxl.load_workbook("testcase.xlsx")
    login_sheet = testcase_wb["Log_in"]
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
    driver.maximize_window()

    for i, row in data_df.iterrows():
        tc_id = row.get('Test case ID')
        if pd.isna(tc_id):
            continue
        try:
            driver.get("http://127.0.0.1:8000/")
            time.sleep(1)
            driver.find_element(By.XPATH, "//*[@class='btn btn-primary']").click()
            time.sleep(2) 

            username = row.get('Username')
            if username and str(username).strip():
                username_box = driver.find_element(By.XPATH, "//*[@name='username']")
                username_box.click()
                username_box.send_keys(str(username).strip())

            password = row.get('Password')
            if password and str(password).strip():
                password_box = driver.find_element(By.XPATH, "//*[@name='password']")
                password_box.click()
                password_box.send_keys(str(password).strip())

            # Nhấn nút "Đăng nhập"
            driver.find_element(By.XPATH, "//*[@class='btn btn-primary']").click()
            time.sleep(2) 

            actual_result = ""

            #Bắt lỗi từ màn hình
            for field in ["username", "password"]:
                try:
                    input_el = driver.find_element(By.NAME, field)
                    msg = driver.execute_script("return arguments[0].validationMessage;", input_el)
                    if msg and msg.strip():
                        actual_result = msg.strip()
                        break
                except:
                    pass
            
            selectors = ["span[style*='color:red']","div[role='alert']"]            
            for selector in selectors:
                try:
                    elements = driver.find_elements(By.CSS_SELECTOR, selector)
                    for el in elements:
                        if el.is_displayed():
                            text = el.text.strip()
                            if text:
                                actual_result = text 
                                break
                except:
                    continue

            # Kiểm tra nếu đăng nhập thành công 
            logged_in = False
            try:
                logout_btn = driver.find_element(By.XPATH, "//*[@class='btn btn-success']")
                if logout_btn.is_displayed():
                    logged_in = True
                    if not actual_result.strip():
                        actual_result = "Không hiển thị thông báo lỗi. Đăng nhập thành công"
                    logout_btn.click()
                    time.sleep(1)
            except:
                pass

            # ==== Cập nhật kết quả vào sheet ====
            for r in range(1, login_sheet.max_row + 1):
                cell_value = login_sheet.cell(r, 1).value
                if cell_value and str(cell_value).strip() == str(tc_id).strip():
                    login_sheet.cell(r, 6).value = actual_result  
                    expected = login_sheet.cell(r, 5).value  
                    if compare_actual_expected(actual_result, expected):
                        login_sheet.cell(r, 7).value = "Pass"
                    else:
                        login_sheet.cell(r, 7).value = "Fail"
                    break  

        except Exception as e:
            actual_result = f"Exception during test: {str(e)}"
            for r in range(1, login_sheet.max_row + 1):
                if login_sheet.cell(r, 1).value and str(login_sheet.cell(r, 1).value).strip() == str(tc_id).strip():
                    login_sheet.cell(r, 6).value = actual_result
                    login_sheet.cell(r, 7).value = "Fail"
                    break

    testcase_wb.save("testcase.xlsx")
    testcase_wb.close()
    driver.quit()
    print("Kiểm thử đăng nhập hoàn tất. Đã cập nhật Kết quả thực sự và Trạng thái vào file testcase.xlsx.")

# HÀM CHUẨN HÓA VÀ SO SÁNH 
def normalize(text):
    if not text:
        return ""
    text = str(text).lower()
    text = ''.join(c for c in text if c not in string.punctuation)
    return ' '.join(text.split())

def compare_actual_expected(actual, expected):
    if not actual or not expected:
        return actual == expected
    norm_actual, norm_expected = normalize(actual), normalize(expected)
    ratio = SequenceMatcher(None, norm_actual, norm_expected).ratio()
    if ratio > 0.7:
        return True

    expected_keywords = set(re.findall(r'\w+', norm_expected))
    actual_keywords = set(re.findall(r'\w+', norm_actual))
    common_keywords = expected_keywords.intersection(actual_keywords)
    key_error_terms = ["hiển thị lỗi", "username", "không hợp lệ", "password", "đăng ký thành công", 
        "không hiển thị lỗi", "đã tồn tại", "không khớp", "nhập lại không khớp", 
        "phải có từ", "đúng định dạng", "vui lòng nhập", "có nhiều nhất", 
        "phải là các chữ", "phải có 10 số", "Ten"]
    for term in key_error_terms:
        if term in norm_expected and term in norm_actual:
            return True
    if expected_keywords and len(common_keywords) / len(expected_keywords) > 0.5:
        return True
    return False

# MENU CHỌN
print("Chọn màn hình kiểm thử:")
print("1: Màn hình Đăng nhập ")
print("2: Màn hình Đăng ký ")
choice = input("Nhập 1 hoặc 2: ").strip()

if choice == '1':
    run_login_tests()
elif choice == '2':
    run_signin_tests()
else:
    print("Lựa chọn không hợp lệ.")

Chọn màn hình kiểm thử:
1: Màn hình Đăng nhập 
2: Màn hình Đăng ký 


Nhập 1 hoặc 2:  2


Kiểm thử đăng ký hoàn tất. Đã cập nhật Kết quả thực sự và Trạng thái vào file testcase.xlsx.
